# pymfx — Quickstart

This notebook walks through the core pymfx workflow:
1. Parse a `.mfx` file
2. Inspect the data
3. Validate against V01–V21 rules
4. Modify and write back
5. Verify checksums

In [ ]:
import sys
sys.path.insert(0, '..')  # if running from notebooks/ folder

import pymfx
print(f"pymfx version: {pymfx.__version__}")

## 1. Parse a .mfx file

In [ ]:
mfx = pymfx.parse("../tests/example_minimal.mfx")
print(type(mfx))

## 2. Inspect the metadata

In [ ]:
m = mfx.meta
print(f"ID           : {m.id}")
print(f"Drone        : {m.drone_id} ({m.drone_type})")
print(f"Pilot        : {m.pilot_id}")
print(f"Start        : {m.date_start}")
print(f"End          : {m.date_end}")
print(f"Duration     : {m.duration_s}s")
print(f"Status       : {m.status}")
print(f"Application  : {m.application}")
print(f"Location     : {m.location}")
print(f"Sensors      : {m.sensors}")
print(f"Data level   : {m.data_level}")
print(f"License      : {m.license}")

## 3. Inspect the trajectory

In [ ]:
traj = mfx.trajectory
print(f"frequency_hz : {traj.frequency_hz}")
print(f"Points       : {len(traj.points)}")
print(f"Checksum     : {traj.checksum}")
print()
print("Schema fields:")
for f in traj.schema_fields:
    print(f"  {f.name}: {f.type}  constraints={f.constraints}")
print()
print("Points:")
for p in traj.points:
    print(f"  t={p.t:.3f}  lat={p.lat}  lon={p.lon}  alt={p.alt_m}m  speed={p.speed_ms}m/s")

## 4. Inspect events

In [ ]:
if mfx.events:
    print(f"Events: {len(mfx.events.events)}")
    for e in mfx.events.events:
        print(f"  t={e.t:.3f}  type={e.type}  severity={e.severity}  detail={e.detail}")
else:
    print("No events section.")

## 5. Inspect the index

In [ ]:
if mfx.index:
    print(f"bbox      : {mfx.index.bbox}")
    print(f"anomalies : {mfx.index.anomalies}")
else:
    print("No index section.")

## 6. Validate

In [ ]:
raw_text = open("../tests/example_minimal.mfx").read()
result = pymfx.validate(mfx, raw_text=raw_text)
print(result)

In [ ]:
# Access errors and warnings separately
print(f"Errors   : {len(result.errors)}")
print(f"Warnings : {len(result.warnings)}")

for issue in result.issues:
    print(f"  {issue}")

## 7. Trigger a validation error on purpose

In [ ]:
# Break the checksum
mfx_broken = pymfx.parse("../tests/example_minimal.mfx")
mfx_broken.trajectory.checksum = "sha256:0000000000000000"

result_broken = pymfx.validate(mfx_broken)
print(result_broken)

## 8. Modify and write back

In [ ]:
mfx_out = pymfx.parse("../tests/example_minimal.mfx")

# Update a field
mfx_out.meta.contact = "updated@lab.fr"
mfx_out.meta.producer = "pymfx"
mfx_out.meta.producer_version = "1.0.0"

# Write — checksums are auto-computed
output_text = pymfx.write(mfx_out)
print(output_text[:800])

In [ ]:
# Validate the output
mfx_reloaded = pymfx.parse(output_text)
result_out = pymfx.validate(mfx_reloaded, raw_text=output_text)
print(result_out)

## 9. Verify checksums manually

In [ ]:
from pymfx import compute_checksum, verify_checksum

traj = mfx_reloaded.trajectory
computed = compute_checksum(traj.raw_lines)
print(f"Computed : {computed}")
print(f"Declared : {traj.checksum}")
print(f"Match    : {verify_checksum(traj.raw_lines, traj.checksum)}")